In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    HAS_MAMBA = True
except ImportError:
    HAS_MAMBA = False

import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Mamba CUDA: {HAS_MAMBA}')

In [ ]:
X_TRAIN_PATH = 'v2_1024/X_train.npy'
Y_TRAIN_PATH = 'v2_1024/y_train.npy'
X_TEST_PATH  = 'v2_1024/X_test.npy'
Y_TEST_PATH  = 'v2_1024/y_test.npy'
X_VAL_PATH   = 'v2_1024/X_val.npy'
Y_VAL_PATH   = 'v2_1024/y_val.npy'

CLASS_NAMES = ['0', '1', '2', '3', '4', '5', '6']

CFG = dict(
    in_channels  = 1,
    num_classes  = 7,  #kiek klasiu
    seq_len      = 1024, # ivesties eilutes ilgis
    patch_size   = 64, 
    dim          = 192,
    dim_inner    = 384,
    dim_state    = 16,
    dim_delta    = 16,
    depth        = 8, #6
    grid_size    = 5,
    spline_order = 3,
)

EPOCHS          = 20
BATCH_SIZE      = 32
LR              = 3e-4
WEIGHT_DECAY    = 1e-4
LABEL_SMOOTHING = 0.1
WARMUP_EPOCHS   = 3
PATIENCE        = 7
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

In [ ]:
class KANLinear(nn.Module):
 def __init__(self, in_features, out_features, grid_size=5, spline_order=3, w_basic=1.0, w_spline=1.0, grid_range=[-1.0, 1.0]):
     super().__init__()
     self.in_features = in_features
     self.out_features = out_features
     self.grid_size = grid_size
     self.spline_order = spline_order
     self.grid_range = grid_range

     self.w_basic = nn.Parameter(torch.tensor(w_basic))
     self.w_spline = nn.Parameter(torch.tensor(w_spline))

     h = (grid_range[1] - grid_range[0]) / grid_size
     grid = torch.arange(-spline_order, grid_size + spline_order + 1, dtype=torch.float32) * h + grid_range[0]
     self.register_buffer("grid", grid)

     self.base_weight   = nn.Parameter(torch.empty(out_features, in_features))
     self.spline_weight = nn.Parameter(torch.empty(out_features, in_features, grid_size + spline_order))
     nn.init.kaiming_uniform_(self.base_weight,   a=math.sqrt(5))
     nn.init.kaiming_uniform_(self.spline_weight, a=math.sqrt(5))

 def b_splines(self, x):
     x = x.unsqueeze(-1)
     bases = ((x >= self.grid[:-1]) & (x < self.grid[1:])).to(x.dtype)
     for k in range(1, self.spline_order + 1):
         left_den = self.grid[k:-1] - self.grid[:-k-1]
         right_den = self.grid[k+1:] - self.grid[1:-k]

         left = (x - self.grid[:-k-1]) / torch.where(left_den == 0, torch.ones_like(left_den), left_den)
         right = (self.grid[k+1:] - x) / torch.where(right_den == 0, torch.ones_like(right_den), right_den)

         bases = left * bases[..., :-1] + right * bases[..., 1:]
     return bases
 def forward(self, x):
     shape = x.shape
     x_flat = x.view(-1, self.in_features)

     base_out = F.linear(F.silu(x_flat), self.base_weight)

     x_clamped = torch.clamp(x_flat, self.grid_range[0] + 1e-4, self.grid_range[1] - 1e-4)
     spline_basis = self.b_splines(x_clamped)

     spline_out = torch.einsum('bik,oik->bo', spline_basis, self.spline_weight)

     out = self.w_basic * base_out + self.w_spline * spline_out
     return out.view(*shape[:-1], self.out_features)

def selective_scan_native(u, delta, A, B, C, D):
    B_batch, L, E = u.shape
    delta_A = torch.exp(delta.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))
    BX      = (delta.unsqueeze(-1) * B.unsqueeze(2)) * u.unsqueeze(-1)
    h  = torch.zeros(B_batch, E, A.shape[1], device=u.device, dtype=u.dtype)
    ys = []
    for t in range(L):
        h = delta_A[:, t] * h + BX[:, t]
        ys.append(torch.einsum('ben,bn->be', h, C[:, t]))
    return torch.stack(ys, dim=1) + u * D.unsqueeze(0).unsqueeze(0)

class SSMPath(nn.Module):
    def __init__(self, d_model, d_state, d_delta):
        super().__init__()
        self.linear_B = nn.Linear(d_model, d_state, bias=False)
        self.linear_C = nn.Linear(d_model, d_state, bias=False)
        self.linear_delta = nn.Linear(d_model, d_delta, bias=False)
        self.dt_proj = nn.Linear(d_delta, d_model, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).repeat(d_model, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        B_mat = self.linear_B(x)
        C_mat = self.linear_C(x)
        delta = self.dt_proj(self.linear_delta(x))
        A_mat = -torch.exp(self.A_log.float())

        if HAS_MAMBA and x.is_cuda:
            try:
                y = selective_scan_fn(
                    x.transpose(1, 2),
                    delta.transpose(1, 2),
                    A_mat,
                    B_mat.transpose(1, 2),
                    C_mat.transpose(1, 2),
                    self.D.float(),
                    z=None,
                    delta_bias=None,
                    delta_softplus=True,
                    return_last_state=False,
                )
                return y.transpose(1, 2)
            except Exception:
                pass

        return selective_scan_native(x, F.softplus(delta), A_mat, B_mat, C_mat, self.D.float())

class VibrMambaBlock(nn.Module):
    def __init__(self, dim, dim_inner, dim_state, dim_delta, grid_size=5, spline_order=3):
        super().__init__()
        self.norm = nn.LayerNorm(dim)

        self.kan_x = KANLinear(dim, dim_inner, grid_size, spline_order)
        self.kan_z = KANLinear(dim, dim_inner, grid_size, spline_order)

        self.forward_ssm = SSMPath(dim_inner, dim_state, dim_delta)
        self.backward_ssm = SSMPath(dim_inner, dim_state, dim_delta)

        self.kan_out = KANLinear(dim_inner, dim, grid_size, spline_order)
    def forward(self, x):
        residual = x
        x = self.norm(x)

        z = self.kan_z(x)
        x = self.kan_x(x)

        y_fwd = self.forward_ssm(x) * F.silu(z)
        y_bwd = torch.flip(self.backward_ssm(torch.flip(x, dims=[1])), dims=[1]) * F.silu(z)

        return self.kan_out(y_fwd + y_bwd) + residual

class PatchEmbedding(nn.Module):
    def __init__(self, seq_len, patch_size, in_channels, dim):
        super().__init__()
        self.num_patches = seq_len // patch_size
        self.proj = nn.Conv1d(in_channels, dim, kernel_size=patch_size, stride=patch_size)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, dim))

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        x = self.proj(x).transpose(1, 2)
        B = x.shape[0]
        cls = self.cls_token.expand(B, -1, -1)
        mid = self.num_patches // 2
        x = torch.cat([x[:, :mid], cls, x[:, mid:]], dim=1)
        return x + self.pos_embed, mid

class VibrMamba(nn.Module):
    def __init__(
        self,
        in_channels=1,
        num_classes=7,
        seq_len=2048,
        patch_size=16,
        dim=192,
        dim_inner=384,
        dim_state=16,
        dim_delta=16,
        depth=6,
        grid_size=5,
        spline_order=3,
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(seq_len, patch_size, in_channels, dim)
        self.blocks = nn.ModuleList(
            [VibrMambaBlock(dim, dim_inner, dim_state, dim_delta, grid_size, spline_order)
             for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        x, mid_idx = self.patch_embed(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, mid_idx])

In [ ]:
X_train = np.load(X_TRAIN_PATH)
y_train = np.load(Y_TRAIN_PATH)
X_test  = np.load(X_TEST_PATH)
y_test  = np.load(Y_TEST_PATH)
X_val   = np.load(X_VAL_PATH)
y_val   = np.load(Y_VAL_PATH)


if X_train.ndim == 2:
    X_train = X_train[:, None, :]
    X_test  = X_test[:,  None, :]
    X_val  = X_val[:,  None, :]

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val:  {X_val.shape}   y_val:  {y_val.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')
print(f'Classes: {np.unique(y_train)}')

train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long)
)
test_ds  = TensorDataset(
    torch.tensor(X_test,  dtype=torch.float32),
    torch.tensor(y_test,  dtype=torch.long))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

In [ ]:
LABEL_SMOOTHING = 0.1
PATIENCE = 4
model = VibrMamba(**CFG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params / 1e3:.1f} K  ({n_params:,})')

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                               weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print(f'Loss: CrossEntropy (label_smoothing={LABEL_SMOOTHING})')
print(f'Scheduler: cosine annealing, warmup={WARMUP_EPOCHS} epochs')
print(f'Early stopping patience: {PATIENCE} epochs')

In [ ]:
from tqdm.notebook import tqdm

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]', leave=False)
    
    for X_batch, y_batch in pbar:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item() * len(y_batch)
        correct    += (logits.argmax(1) == y_batch).sum().item()
        total      += len(y_batch)
        pbar.set_postfix(loss=f'{loss.item():.4f}',
                         acc=f'{correct/total:.4f}')
    train_loss = total_loss / total
    train_acc  = correct   / total

    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in tqdm(val_loader, desc=f'Epoch {epoch}/{EPOCHS} [Val]', leave=False):
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits     = model(X_batch)
            loss       = criterion(logits, y_batch)
            total_loss += loss.item() * len(y_batch)
            correct    += (logits.argmax(1) == y_batch).sum().item()
            total      += len(y_batch)
    val_loss = total_loss / total
    val_acc  = correct   / total

    scheduler.step(val_acc)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch:3d}/{EPOCHS}  '
          f'train loss {train_loss:.4f}  acc {train_acc:.4f}  '
          f'val loss {val_loss:.4f}  acc {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), 'vibrmamba_PSD_randp.pt')
        print(f'  ↑ New best val acc: {val_acc:.4f} — model saved.')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).')
            break

print(f'\nBest val accuracy: {best_val_acc:.4f}')
print('Best model saved to vibrmamba_gear_raw_1024.pt')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], label='Train')
axes[0].plot(epochs_range, history['val_loss'],   label='Validation')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs_range, history['train_acc'], label='Train')
axes[1].plot(epochs_range, history['val_acc'],   label='Validation')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc = accuracy_score(all_labels, all_preds)
print(f'Test accuracy: {acc:.4f} ({acc*100:.2f}%)')
print()
print(classification_report(all_labels, all_preds,
                             target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Klasifikavimo matrica (kiekiai)')
axes[0].set_xlabel('Prognozuota'); axes[0].set_ylabel('Tikroji')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            vmin=0, vmax=1, ax=axes[1])
axes[1].set_title('Klasifikavimo matrica (normalizuota)')
axes[1].set_xlabel('Prognozuota'); axes[1].set_ylabel('Tikroji')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()